## Imports 

In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow as tf
from sklearn.preprocessing import StandardScaler


## Prepare data
Here we load the data, but also experimented with some additional variables.
The "cols_to_remove" is used in order to select relevant features for our model 

In [ ]:

# Load data 
data = pd.read_csv("./../2_BaselineModel/data_after_imputation.csv")
data['Datum'] = pd.to_datetime(data['Datum'])

#Here we add some additional features that might be useful for the model

#Add day of the week column (Monday=1, Sunday=7)
data['Weekday'] = data['Datum'].dt.weekday + 1

#Make the weekday one hot encoded
#data = pd.get_dummies(data, columns=['Weekday'], prefix='Wochentag', drop_first=False, dtype=int) 

#There is a trend on Saturdays and Sundays for some products, so we add separate binary columns for Saturday and Sunday
data['is_saturday'] = (data['Weekday'] == 6).astype(int)
data['is_sunday'] = (data['Weekday'] == 7).astype(int)

#Days until Silvester (New Year's Eve)
data['days_to_silvester'] = (pd.to_datetime(data['Datum'].dt.year.astype(str) + '-12-31') - data['Datum']).dt.days


#Variable to descrive the seasonal bread selling trend, more on December than in November, nearly none in other months
data['month'] = data['Datum'].dt.month          # Extract month from the date

def seasonal_score(month):
    if month == 12:
        return 2  # higher score for December
    elif month == 11:
        return 1  # lower score for November
    else:
        return 0  # zero outside Nov & Dec

data['seasonal_bread_score'] = data['month'].apply(seasonal_score)


# Check column names to decide what to remove
print(data.columns)

# Remove columns that are not needed for the model, comment out if you want to keep them
cols_to_remove = [
    'Wochenende',
    'Silvester',
    'Bewoelkung',
    #'Temperatur',
    'Windgeschwindigkeit',
    'KielerWoche',
    #'Schulferien',
    'Feiertage',
    'Advent',
    'Wahltag', 
    'VPI',
    'Niederschlag',
    'mask_Temperatur_Windgeschwindigkeit',
    'mask_Bewoelkung',
    'Wettercode_0',  
    'Wettercode_10',
    'Wettercode_21',
    'Wettercode_5',
    'Wettercode_61',
    'Wettercode_63',
    'Wettercode_nan',
    'Wettercode_rest',
    #'Warengruppe_1',
    #'Warengruppe_2',
    #'Warengruppe_3',
    #'Warengruppe_4',
    #'Warengruppe_5',
    #'Warengruppe_6',
    #'Weekday',
    #'days_to_silvester',
    'is_saturday',
    'is_sunday',
    'month', 
    'seasonal_score'
]

# Drop the columns that are not needed for the model
data = data.drop(columns=cols_to_remove)

#Split the data into training, validation, and test sets
training_df = data[(data["Datum"] >= "2013-07-01") & (data["Datum"] <= "2017-07-31")]
validation_df = data[(data["Datum"] >= "2017-08-01") & (data["Datum"] <= "2018-07-31")]
test_df = data[(data["Datum"] >= "2018-08-01") & (data["Datum"] <= "2019-07-31")]

print("\nTraining set shape:", training_df.shape)
print("Validation set shape:", validation_df.shape)
print("Test set shape:", test_df.shape)

# Separate features 
training_features = training_df.drop(["id", "Datum", "Umsatz"], axis=1)
validation_features = validation_df.drop(["id", "Datum", "Umsatz"], axis=1)
test_features = test_df.drop(["id", "Datum", "Umsatz"], axis=1)

# Separate labels
training_labels = training_df[["Umsatz"]]
validation_labels = validation_df[["Umsatz"]]
test_labels = test_df[["id", "Umsatz"]]

# Print dimensions of the dataframes
print("Training features dimensions:", training_features.shape)
print("Validation features dimensions:", validation_features.shape)
print("Test features dimensions:", test_features.shape)
print("Training labels dimensions:", training_labels.shape)
print("Validation labels dimensions:", validation_labels.shape)
print("Test labels dimensions:", test_labels.shape)

# Create subdirectory for the pickle files
subdirectory = "GA_pickle_data"
os.makedirs(subdirectory, exist_ok=True)

# Export the prepared data to subdirectory as pickle files
training_features.to_pickle(f"{subdirectory}/training_features.pkl")
validation_features.to_pickle(f"{subdirectory}/validation_features.pkl")
test_features.to_pickle(f"{subdirectory}/test_features.pkl")
training_labels.to_pickle(f"{subdirectory}/training_labels.pkl")
validation_labels.to_pickle(f"{subdirectory}/validation_labels.pkl")
test_labels.to_pickle(f"{subdirectory}/test_labels.pkl")

#See colnames of the the pickle files from above
print("\nColumn names in training features:", training_features.columns.tolist())
print("Column names in validation features:", validation_features.columns.tolist())
print("Column names in test features:", test_features.columns.tolist())
print("Column names in training labels:", training_labels.columns.tolist())
print("Column names in validation labels:", validation_labels.columns.tolist())
print("Column names in test labels:", test_labels.columns.tolist())


In [ ]:
#Read data

# Define the file paths
subdirectory = "GA_pickle_data"
training_features_path = f"{subdirectory}/training_features.pkl"
validation_features_path = f"{subdirectory}/validation_features.pkl"
test_features_path = f"{subdirectory}/test_features.pkl"
training_labels_path = f"{subdirectory}/training_labels.pkl"
validation_labels_path = f"{subdirectory}/validation_labels.pkl"
test_labels_path = f"{subdirectory}/test_labels.pkl"

# Read the pickle files
training_features = pd.read_pickle(training_features_path)
validation_features = pd.read_pickle(validation_features_path)
test_features = pd.read_pickle(test_features_path)
training_labels = pd.read_pickle(training_labels_path)
validation_labels = pd.read_pickle(validation_labels_path)
test_labels = pd.read_pickle(test_labels_path)

#print dimensions of the dataframes
print("Training features dimensions:", training_features.shape)
print("Validation features dimensions:", validation_features.shape)
print("Test features dimensions:", test_features.shape)
print("Training labels dimensions:", training_labels.shape)
print("Validation labels dimensions:", validation_labels.shape)
print("Test labels dimensions:", test_labels.shape)

## Data Normalization

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

training_features = scaler.fit_transform(training_features)
validation_features = scaler.transform(validation_features)
test_features = scaler.transform(test_features)

In [ ]:
#Defining the Neural Network

model = Sequential([
    InputLayer(shape=(training_features.shape[1],)), 
    Dense(128, activation='relu',  kernel_regularizer=l2(0.0001)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.summary()

In [ ]:
#Compiling and Training the Model
#we  compile the model using Mean Squared Error (MSE) as the loss function and Adam optimizer. 
#the model is then trained using the training data.

model.compile(loss="mse", optimizer=Adam(learning_rate=0.0001))

#we use early stop to stop training when the validation loss doesn't improve for a set number of epochs (patience).
#this is to prevent overfitting and sacv training time
early_stop = EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True)

history = model.fit(
	training_features, training_labels,
	epochs=600,
	batch_size=32,
	validation_data=(validation_features, validation_labels),
	callbacks=[early_stop],
	verbose=1
)

In [ ]:
#Save the model

model.save("python_model_.keras")

In [ ]:
#Plotting training history 

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
#Make predictions and Evaluate the Model

import numpy as np

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_mask = y_true != 0
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

training_predictions = model.predict(training_features)
validation_predictions = model.predict(validation_features)


print(f"MAPE on the Training Data: {mape(training_labels, training_predictions):.2f}%")
print(f"MAPE on the Validation Data: {mape(validation_labels, validation_predictions):.2f}%")

In [ ]:
#Calculate MAPE for the different warengruppe

import numpy as np
import pandas as pd

# Step 1: Recreate DataFrame for training and validation features (if not already)
feature_columns = training_df.drop(["id", "Datum", "Umsatz"], axis=1).columns.tolist()

X_train_df = pd.DataFrame(training_features, columns=feature_columns)
X_val_df = pd.DataFrame(validation_features, columns=feature_columns)

# Step 2: Function to extract Warengruppe (1–6)
def extract_warengruppe(df):
    warengruppe_cols = [col for col in df.columns if col.startswith("Warengruppe_")]
    return df[warengruppe_cols].idxmax(axis=1).str.extract(r'_(\d+)')[0].astype(int)

# Step 3: Compute MAPE per group
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_mask = y_true != 0
    return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100

def mape_per_group(features_df, labels_df, predictions):
    groups = extract_warengruppe(features_df)
    df = pd.DataFrame({
        "Umsatz": labels_df.values.flatten(),
        "Predicted": predictions.flatten(),
        "Warengruppe": groups
    })
    return df.groupby("Warengruppe").apply(lambda g: mape(g["Umsatz"], g["Predicted"]))

# Step 4: Run it
training_mape_by_group = mape_per_group(X_train_df, training_labels, training_predictions)
validation_mape_by_group = mape_per_group(X_val_df, validation_labels, validation_predictions)

print("MAPE per Warengruppe – Training:")
print(training_mape_by_group)
print("\nMAPE per Warengruppe – Validation:")
print(validation_mape_by_group)


In [ ]:
#Visualizing Predictions vs Actual Values for training and validaiton data
# and for the different product groups

#Plotting per warengruppe       
import matplotlib.pyplot as plt

def plot_predictions_by_group(df, title_prefix):
    groups = sorted(df['Warengruppe'].unique())
    num_groups = len(groups)
    nrows, ncols = 2, 3 

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 8), sharey=True)
    axes = axes.flatten() 

    if num_groups == 1:
        axes = [axes]  # ensure iterable if only 1 subplot

    for ax, group in zip(axes, groups):
        group_df = df[df['Warengruppe'] == group].reset_index(drop=True).head(200)
        ax.plot(group_df['actual'], label='Actual', color='red')
        ax.plot(group_df['prediction'], label='Predicted', color='blue')
        ax.set_title(f'{title_prefix} – Gruppe {group}')
        ax.set_xlabel('Day')
        ax.set_ylabel('Umsatz')
        ax.legend()

    plt.tight_layout()
    plt.show()


    # Create training DataFrame with Warengruppe
train_df = pd.DataFrame({
    'prediction': training_predictions,
    'actual': training_labels,
})
train_df['Warengruppe'] = extract_warengruppe(X_train_df)

# Create validation DataFrame with Warengruppe
val_df = pd.DataFrame({
    'prediction': validation_predictions,
    'actual': validation_labels,
})
val_df['Warengruppe'] = extract_warengruppe(X_val_df)

# Plot
plot_predictions_by_group(train_df, 'Training')
plot_predictions_by_group(val_df, 'Validation')


In [ ]:
#Predict on the test data

# Example: Predict on test_features
test_labels["Umsatz"] = model.predict(test_features)

test_labels[["id", "Umsatz"]].to_csv("nn_submission_.csv", index=False)
